In [4]:
import pandas as pd

bls = pd.read_csv('bls_processed.csv', dtype={'OCC_CODE': str})

print("Shape:", bls.shape)
print("\nAll column names:")
print(bls.columns.tolist())
print("\nUnique values in YEAR column (if exists):")
if 'YEAR' in bls.columns:
    print(bls['YEAR'].value_counts())
else:
    print("NO YEAR COLUMN FOUND")
    # Check for similar column names
    year_like = [c for c in bls.columns if 'year' in c.lower() or 'yr' in c.lower()]
    print("Year-like columns:", year_like)

print("\nFirst 2 rows:")
print(bls.head(2).to_string())

Shape: (2450, 41)

All column names:
['AREA', 'AREA_TITLE', 'AREA_TYPE', 'NAICS', 'NAICS_TITLE', 'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY', 'YEAR', 'PRIM_STATE', 'PCT_RPT', 'MAJOR_SOC', 'MAJOR_TITLE', 'wage_eqw', 'wage_eqf', 'wage_entropy', 'wage_cluster', 'is_random_sample_10pct', 'PC1']

Unique values in YEAR column (if exists):
YEAR
2024    831
2022    830
2019    789
Name: count, dtype: int64

First 2 rows:
   AREA AREA_TITLE  AREA_TYPE  NAICS     NAICS_TITLE         I_GROUP  OWN_CODE OCC_CODE                        OCC_TITLE   O_GROUP  TOT_EMP  EMP_PRSE  JOBS_1000  LOC_QUOTIENT  PCT_TOTAL  H_MEAN    A_MEAN  MEAN_PRSE  H_PCT10  H_PCT25  H_MEDIAN  H_PCT75  H_PCT90  A_PCT10   A_PCT25  A_MEDIAN   A_PCT75  A_PCT90 ANNUAL HOURLY  YEAR PRIM_STATE

In [5]:
import pandas as pd

bls = pd.read_csv('bls_processed.csv', dtype={'OCC_CODE': str})

# ── 1. Keep only needed columns ─────────────────────────────────
keep = ['OCC_CODE', 'OCC_TITLE', 'TOT_EMP', 
        'A_MEDIAN', 'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90', 'YEAR']
bls = bls[keep].copy()

# ── 2. Filter to detailed occupations (XX-XXXX format) ──────────
bls = bls[bls['OCC_CODE'].str.match(r'^\d{2}-\d{4}$')]

# ── 3. Clean numeric columns ────────────────────────────────────
num_cols = ['TOT_EMP', 'A_MEDIAN', 'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90']
for col in num_cols:
    bls[col] = pd.to_numeric(
        bls[col].astype(str)
                .str.replace(',', '', regex=False)
                .str.replace('*', '', regex=False)
                .str.strip(),
        errors='coerce'
    )

# ── 4. Inflation adjust to real 2024 dollars ────────────────────
cpi = {2019: 1.163, 2022: 1.081, 2024: 1.000}
bls['real_median_wage'] = bls['A_MEDIAN'] * bls['YEAR'].map(cpi)

# ── 5. Pivot to wide format ─────────────────────────────────────
bls_wide = bls.pivot_table(
    index='OCC_CODE',
    columns='YEAR',
    values=['TOT_EMP', 'real_median_wage',
            'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90']
)
bls_wide.columns = [f"{col}_{int(yr)}" for col, yr in bls_wide.columns]
bls_wide = bls_wide.reset_index()

# ── 6. Add occupation title ─────────────────────────────────────
titles = (bls[bls['YEAR'] == 2024][['OCC_CODE', 'OCC_TITLE']]
          .drop_duplicates('OCC_CODE'))
bls_wide = bls_wide.merge(titles, on='OCC_CODE', how='left')

# ── 7. Compute change variables ─────────────────────────────────
bls_wide['emp_change_pct_19_24'] = (
    (bls_wide['TOT_EMP_2024'] - bls_wide['TOT_EMP_2019'])
    / bls_wide['TOT_EMP_2019'] * 100
)
bls_wide['emp_change_pct_19_22'] = (
    (bls_wide['TOT_EMP_2022'] - bls_wide['TOT_EMP_2019'])
    / bls_wide['TOT_EMP_2019'] * 100
)
bls_wide['emp_change_pct_22_24'] = (
    (bls_wide['TOT_EMP_2024'] - bls_wide['TOT_EMP_2022'])
    / bls_wide['TOT_EMP_2022'] * 100
)
bls_wide['wage_change_pct_19_24'] = (
    (bls_wide['real_median_wage_2024'] - bls_wide['real_median_wage_2019'])
    / bls_wide['real_median_wage_2019'] * 100
)
bls_wide['wage_change_pct_19_22'] = (
    (bls_wide['real_median_wage_2022'] - bls_wide['real_median_wage_2019'])
    / bls_wide['real_median_wage_2019'] * 100
)
bls_wide['wage_change_pct_22_24'] = (
    (bls_wide['real_median_wage_2024'] - bls_wide['real_median_wage_2022'])
    / bls_wide['real_median_wage_2022'] * 100
)

# ── 8. Verify ───────────────────────────────────────────────────
print(f"Shape: {bls_wide.shape}")
print(f"Columns: {bls_wide.columns.tolist()}")
print(f"Missing values: {bls_wide.isnull().sum().sum()}")
print(f"\nSample row (Software Developers):")
print(bls_wide[bls_wide['OCC_CODE'] == '15-1252'].to_string())

# ── 9. Save ─────────────────────────────────────────────────────
bls_wide.to_csv('bls_cleaned.csv', index=False)
print("\nSaved: bls_cleaned.csv")

Shape: (861, 26)
Columns: ['OCC_CODE', 'A_PCT10_2019', 'A_PCT10_2022', 'A_PCT10_2024', 'A_PCT25_2019', 'A_PCT25_2022', 'A_PCT25_2024', 'A_PCT75_2019', 'A_PCT75_2022', 'A_PCT75_2024', 'A_PCT90_2019', 'A_PCT90_2022', 'A_PCT90_2024', 'TOT_EMP_2019', 'TOT_EMP_2022', 'TOT_EMP_2024', 'real_median_wage_2019', 'real_median_wage_2022', 'real_median_wage_2024', 'OCC_TITLE', 'emp_change_pct_19_24', 'emp_change_pct_19_22', 'emp_change_pct_22_24', 'wage_change_pct_19_24', 'wage_change_pct_19_22', 'wage_change_pct_22_24']
Missing values: 1624

Sample row (Software Developers):
   OCC_CODE  A_PCT10_2019  A_PCT10_2022  A_PCT10_2024  A_PCT25_2019  A_PCT25_2022  A_PCT25_2024  A_PCT75_2019  A_PCT75_2022  A_PCT75_2024  A_PCT90_2019  A_PCT90_2022  A_PCT90_2024  TOT_EMP_2019  TOT_EMP_2022  TOT_EMP_2024  real_median_wage_2019  real_median_wage_2022  real_median_wage_2024            OCC_TITLE  emp_change_pct_19_24  emp_change_pct_19_22  emp_change_pct_22_24  wage_change_pct_19_24  wage_change_pct_19_22  wage_

In [6]:
import pandas as pd

bls_wide = pd.read_csv('bls_cleaned.csv', dtype={'OCC_CODE': str})

# How many occupations have complete data across all 3 years?
complete = bls_wide.dropna(subset=[
    'TOT_EMP_2019', 'TOT_EMP_2022', 'TOT_EMP_2024',
    'real_median_wage_2019', 'real_median_wage_2022', 'real_median_wage_2024'
])
print(f"Total occupations:              {len(bls_wide)}")
print(f"Complete across all 3 years:    {len(complete)}")
print(f"Missing at least one year:      {len(bls_wide) - len(complete)}")

# Which year has the most missing?
print(f"\nMissing TOT_EMP_2019:  {bls_wide['TOT_EMP_2019'].isnull().sum()}")
print(f"Missing TOT_EMP_2022:  {bls_wide['TOT_EMP_2022'].isnull().sum()}")
print(f"Missing TOT_EMP_2024:  {bls_wide['TOT_EMP_2024'].isnull().sum()}")

print(f"\nMissing real_median_wage_2019: {bls_wide['real_median_wage_2019'].isnull().sum()}")
print(f"Missing real_median_wage_2022: {bls_wide['real_median_wage_2022'].isnull().sum()}")
print(f"Missing real_median_wage_2024: {bls_wide['real_median_wage_2024'].isnull().sum()}")

# Show sample of occupations missing 2019 data
missing_2019 = bls_wide[bls_wide['TOT_EMP_2019'].isnull()][['OCC_CODE', 'OCC_TITLE']]
print(f"\nSample occupations missing 2019 data:")
print(missing_2019.head(10).to_string(index=False))

Total occupations:              861
Complete across all 3 years:    750
Missing at least one year:      111

Missing TOT_EMP_2019:  72
Missing TOT_EMP_2022:  31
Missing TOT_EMP_2024:  30

Missing real_median_wage_2019: 83
Missing real_median_wage_2022: 47
Missing real_median_wage_2024: 52

Sample occupations missing 2019 data:
OCC_CODE                                              OCC_TITLE
 11-2032                              Public Relations Managers
 11-2033                                   Fundraising Managers
 11-3012                       Administrative Services Managers
 11-3013                                    Facilities Managers
 11-9072 Entertainment and Recreation Managers, Except Gambling
 11-9179                   Personal Service Managers, All Other
 11-9199                                    Managers, All Other
 13-1082                         Project Management Specialists
 13-1199             Business Operations Specialists, All Other
 13-2051                      F

In [7]:
import pandas as pd

bls_wide = pd.read_csv('bls_cleaned.csv', dtype={'OCC_CODE': str})

# ── Drop occupations missing any core variable across any year ───
core_cols = [
    'TOT_EMP_2019', 'TOT_EMP_2022', 'TOT_EMP_2024',
    'real_median_wage_2019', 'real_median_wage_2022', 'real_median_wage_2024'
]
bls_final = bls_wide.dropna(subset=core_cols).copy()

# ── Final checks ─────────────────────────────────────────────────
print(f"Occupations kept:    {len(bls_final)}")
print(f"Occupations dropped: {len(bls_wide) - len(bls_final)}")
print(f"Remaining missing:   {bls_final.isnull().sum().sum()}")
print(f"Columns:             {bls_final.columns.tolist()}")

# ── Spot checks ──────────────────────────────────────────────────
print("\nSpot check — Chief Executives (11-1011):")
print(bls_final[bls_final['OCC_CODE'] == '11-1011']
      [['OCC_CODE', 'OCC_TITLE', 'TOT_EMP_2019', 'TOT_EMP_2022', 
        'TOT_EMP_2024', 'real_median_wage_2019', 'real_median_wage_2024',
        'emp_change_pct_19_24', 'wage_change_pct_19_24']].to_string())

print("\nSpot check — Registered Nurses (29-1141):")
print(bls_final[bls_final['OCC_CODE'] == '29-1141']
      [['OCC_CODE', 'OCC_TITLE', 'TOT_EMP_2019', 'TOT_EMP_2024',
        'emp_change_pct_19_24', 'wage_change_pct_19_24']].to_string())

# ── Save final clean file ────────────────────────────────────────
bls_final.to_csv('bls_cleaned.csv', index=False)
print(f"\nSaved: bls_cleaned.csv")
print(f"Final shape: {bls_final.shape}")

Occupations kept:    750
Occupations dropped: 111
Remaining missing:   93
Columns:             ['OCC_CODE', 'A_PCT10_2019', 'A_PCT10_2022', 'A_PCT10_2024', 'A_PCT25_2019', 'A_PCT25_2022', 'A_PCT25_2024', 'A_PCT75_2019', 'A_PCT75_2022', 'A_PCT75_2024', 'A_PCT90_2019', 'A_PCT90_2022', 'A_PCT90_2024', 'TOT_EMP_2019', 'TOT_EMP_2022', 'TOT_EMP_2024', 'real_median_wage_2019', 'real_median_wage_2022', 'real_median_wage_2024', 'OCC_TITLE', 'emp_change_pct_19_24', 'emp_change_pct_19_22', 'emp_change_pct_22_24', 'wage_change_pct_19_24', 'wage_change_pct_19_22', 'wage_change_pct_22_24']

Spot check — Chief Executives (11-1011):
  OCC_CODE         OCC_TITLE  TOT_EMP_2019  TOT_EMP_2022  TOT_EMP_2024  real_median_wage_2019  real_median_wage_2024  emp_change_pct_19_24  wage_change_pct_19_24
0  11-1011  Chief Executives      205890.0      199240.0      211850.0              214526.98               206420.0               2.89475              -3.779003

Spot check — Registered Nurses (29-1141):
    OCC_